# Preprocessing & Dataset Tests
End-to-end validation of `utils/preprocessing.py`, `utils/dataset.py`, and `models/`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import Counter
from torch.utils.data import DataLoader

from utils.preprocessing import (
    load_phenotypic, build_subject_file_map,
    preprocess_volume, load_nifti, get_crop_shape
)
from utils.dataset import build_datasets
from models.single_modal_3d import fmri_cnn, smri_cnn
from models.multi_modal_3d import MultiModal3DCNN

DATA_DIR = '../data/raw'
print('Imports OK')

## 1. Phenotypic CSV

In [ ]:
pheno = load_phenotypic(DATA_DIR)
print(f'Total subjects: {len(pheno)}')
print(f'  ADHD (label=0): {(pheno["label"]==0).sum()}')
print(f'  TDC  (label=1): {(pheno["label"]==1).sum()}')
print(f'Columns: {list(pheno.columns)}')
pheno.head()

## 2. Subject File Map

In [ ]:
file_map = build_subject_file_map(DATA_DIR)
print(f'Subjects with at least one derivative: {len(file_map)}')

complete = [sid for sid, d in file_map.items() if len(d) == 3]
print(f'Subjects with all 3 derivatives: {len(complete)}')

sid = complete[0]
print(f'\nExample subject {sid}:')
for k, v in file_map[sid].items():
    print(f'  {k}: {v}')

## 3. Raw Volume Shapes vs Paper Targets
Paper: fMRI features crop to **47×60×46**, sMRI to **90×117×100** (Section IV-A).
Raw volumes must be at least this large in every dimension.

In [ ]:
sid = complete[0]
for deriv, path in file_map[sid].items():
    vol = load_nifti(path)
    target = get_crop_shape(deriv)
    fits = all(v >= t for v, t in zip(vol.shape, target))
    print(f'{deriv:6s}: raw={vol.shape}  target={target}  fits={fits}  '
          f'min={vol.min():.3f}  max={vol.max():.3f}')

# Check across ALL subjects — flag any that are too small to crop
print('\nChecking all subjects...')
too_small = []
for s in complete:
    for deriv, path in file_map[s].items():
        vol = load_nifti(path)
        target = get_crop_shape(deriv)
        if not all(v >= t for v, t in zip(vol.shape, target)):
            too_small.append((s, deriv, vol.shape, target))
if too_small:
    print(f'  WARNING — {len(too_small)} volumes smaller than crop target:')
    for x in too_small: print(' ', x)
else:
    print(f'  All {len(complete)*3} volumes are large enough to crop.')

## 4. Normalization Check
The paper does not explicitly mention per-subject z-score normalization — DPARSF already
normalizes to MNI space. We apply z-score anyway to remove inter-subject intensity scale
differences. This cell shows pre- vs post-normalization statistics.

In [ ]:
sid = complete[0]
for deriv, path in file_map[sid].items():
    raw = load_nifti(path)
    t   = preprocess_volume(path, deriv).squeeze().numpy()
    print(f'{deriv:6s}  raw: mean={raw.mean():.4f} std={raw.std():.4f} '
          f'  →  normed: mean={t.mean():.4f} std={t.std():.4f}')

## 5. preprocess_volume — Shape Check

In [ ]:
sid = complete[0]
for deriv, path in file_map[sid].items():
    t = preprocess_volume(path, deriv)
    expected = get_crop_shape(deriv)
    shape_ok = tuple(t.shape[1:]) == expected
    print(f'{deriv}: shape={tuple(t.shape)}  shape_ok={shape_ok}  '
          f'mean={t.mean():.4f}  std={t.std():.4f}')

## 6. Consistency Check Across All Subjects

In [ ]:
errors = []
for sid in complete:
    for deriv, path in file_map[sid].items():
        try:
            t = preprocess_volume(path, deriv)
            expected = get_crop_shape(deriv)
            if tuple(t.shape[1:]) != expected:
                errors.append((sid, deriv, f'shape mismatch: {tuple(t.shape)}'))
        except Exception as e:
            errors.append((sid, deriv, str(e)))

if errors:
    print(f'ERRORS ({len(errors)}):')
    for e in errors: print(' ', e)
else:
    print(f'All {len(complete) * 3} volumes passed shape check.')

## 7. Brain Slice Visualisation

In [ ]:
sid = complete[0]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (deriv, path) in zip(axes, file_map[sid].items()):
    t = preprocess_volume(path, deriv).squeeze().numpy()
    mid = t.shape[2] // 2
    ax.imshow(t[:, :, mid].T, cmap='gray', origin='lower')
    ax.set_title(f'{deriv}  (subject {sid})')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Dataset Tests

In [ ]:
ds_falff = build_datasets(DATA_DIR, derivative='falff')
ds_reho  = build_datasets(DATA_DIR, derivative='reho')
ds_gm    = build_datasets(DATA_DIR, derivative='gm')

print(f'fALFF dataset: {len(ds_falff)} subjects')
print(f'ReHo  dataset: {len(ds_reho)} subjects')
print(f'GM    dataset: {len(ds_gm)} subjects')

tensor, label = ds_falff[0]
print(f'\nSample 0 — shape: {tuple(tensor.shape)}, label: {label} ({"ADHD" if label==0 else "TDC"})')

In [ ]:
ds_multi = build_datasets(DATA_DIR, multi_modal=True, fmri_derivative='falff', smri_derivative='gm')
print(f'Multi-modal dataset: {len(ds_multi)} subjects')

fmri, smri, label = ds_multi[0]
print(f'Sample 0 — fMRI: {tuple(fmri.shape)}, sMRI: {tuple(smri.shape)}, label: {label}')

In [ ]:
loader = DataLoader(ds_falff, batch_size=4, shuffle=True)
batch_tensors, batch_labels = next(iter(loader))
print(f'Batch shape:  {tuple(batch_tensors.shape)}')  # expect (4, 1, 47, 60, 46)
print(f'Batch labels: {batch_labels.tolist()}')

In [ ]:
labels_all = [ds_falff[i][1] for i in range(len(ds_falff))]
counts = Counter(labels_all)
print(f'ADHD: {counts[0]}, TDC: {counts[1]}, ADHD%: {100*counts[0]/len(ds_falff):.1f}%')

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(['ADHD', 'TDC'], [counts[0], counts[1]], color=['steelblue', 'salmon'])
ax.set_ylabel('Subjects')
ax.set_title('Class Distribution')
plt.tight_layout()
plt.show()

## 9. Model Shape Checks
Verify the single-modal and multi-modal CNNs produce the right output shapes
and that intermediate feature vectors are 512-dim as specified in the paper.

In [ ]:
fmri_model = fmri_cnn()
smri_model = smri_cnn()
fmri_model.eval()
smri_model.eval()

dummy_fmri = torch.zeros(2, 1, 47, 60, 46)   # batch=2
dummy_smri = torch.zeros(2, 1, 90, 117, 100)

with torch.no_grad():
    out_fmri  = fmri_model(dummy_fmri)
    feat_fmri = fmri_model.get_features(dummy_fmri)
    out_smri  = smri_model(dummy_smri)
    feat_smri = smri_model.get_features(dummy_smri)

print('--- Single-modal fMRI CNN ---')
print(f'  Input:   {tuple(dummy_fmri.shape)}')
print(f'  Feature: {tuple(feat_fmri.shape)}  (expect (2, 512))')
print(f'  Output:  {tuple(out_fmri.shape)}   (expect (2, 2))')

print('\n--- Single-modal sMRI CNN ---')
print(f'  Input:   {tuple(dummy_smri.shape)}')
print(f'  Feature: {tuple(feat_smri.shape)}  (expect (2, 512))')
print(f'  Output:  {tuple(out_smri.shape)}   (expect (2, 2))')

In [ ]:
multi_model = MultiModal3DCNN()
multi_model.eval()

with torch.no_grad():
    out_multi = multi_model(dummy_fmri, dummy_smri)

print('--- Multi-modal CNN ---')
print(f'  fMRI input: {tuple(dummy_fmri.shape)}')
print(f'  sMRI input: {tuple(dummy_smri.shape)}')
print(f'  Output:     {tuple(out_multi.shape)}  (expect (2, 2))')

In [ ]:
# Parameter counts
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'fMRI branch params:       {count_params(fmri_model):,}')
print(f'sMRI branch params:       {count_params(smri_model):,}')
print(f'Multi-modal total params: {count_params(MultiModal3DCNN()):,}')